# 03. Training & Evaluation (Image × Regression Cell)

**Phase 2**: 폴루션 train × 회귀모델 5개 학습 → clean test R² 측정.

회귀 헤드 Linear(·,1) + MSELoss, R²(음수 clip to 0). 체크포인트로 (dataset,polluter,level,model) skip. GPU 필요.

In [1]:
# ============================================================
# 0-1. Drive 마운트 + GPU 확인
# ============================================================
from google.colab import drive
drive.mount('/content/drive')

import os, sys, json
import numpy as np
import pandas as pd
import torch

BASE = '/content/drive/MyDrive/capstone/dsc'
RESULTS_DIR = f'{BASE}/results'
DATA_DIR = f'{BASE}/data/image_regression'
POLLUTED_DIR = f'{BASE}/data/image_regression_polluted'
os.makedirs(RESULTS_DIR, exist_ok=True)
os.makedirs(DATA_DIR, exist_ok=True)

if BASE not in sys.path:
    sys.path.insert(0, BASE)

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'device: {device} | torch: {torch.__version__}')

Mounted at /content/drive
device: cuda | torch: 2.11.0+cu128


In [2]:
# ============================================================
# 0-2. 의존성 설치 (Colab)
# ============================================================
%pip install -q datasets timm imagehash opencv-python-headless

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 296.7/296.7 kB 8.7 MB/s eta 0:00:00


In [3]:
# ============================================================
# 사전등록 메타 (ADR-018) — HuggingFace datasets
# ============================================================
DATASETS = {
    'UTKFace':       {'hf': 'Subh775/UTKFace_demographics_V1', 'target': 'age',          'image_col': 'image', 'role': 'tune'},
    'SCUT_FBP5500':  {'hf': 'MnLgt/scut-fbp5500',              'target': 'beauty_score', 'image_col': 'image', 'role': 'held-out'},
}
TUNE_DS, HELD_DS = 'UTKFace', 'SCUT_FBP5500'
SAMPLE_CAP = 2000   # RAM 절약(02와 일치)   # DSC 계산용 (메모리/시간 절약)
RANDOM_SEED = 42
ML_SPLIT_SEED = 1
ML_TEST_SIZE = 0.2
print(f'데이터셋: {list(DATASETS.keys())} (튜닝={TUNE_DS}, held-out={HELD_DS})')

# Phase 1 축소 (이미지 분류와 동일 정책): 모델 2종 + EPOCHS 10
MODEL_NAMES_FULL = ['ResNet18', 'EfficientNetB0', 'MobileNetV3small', 'ViTTiny', 'CNNSimple']
POLLUTION_LEVELS = [0.1, 0.3, 0.5, 0.7, 0.9]
MODEL_NAMES = ['ResNet18', 'CNNSimple']
EPOCHS = 10
BATCH_SIZE = 128
LR = 1e-3
IMAGE_SIZE = 224
print(f'Phase 1: models={MODEL_NAMES}, EPOCHS={EPOCHS}, levels={POLLUTION_LEVELS}')

데이터셋: ['UTKFace', 'SCUT_FBP5500'] (튜닝=UTKFace, held-out=SCUT_FBP5500)
Phase 1: models=['ResNet18', 'CNNSimple'], EPOCHS=10, levels=[0.1, 0.3, 0.5, 0.7, 0.9]


In [4]:
# ============================================================
# HF 데이터셋 로더 + numpy 변환 (split-first: HF는 train split만 → 자체 분할)
# ============================================================
from datasets import load_dataset

def load_hf_split(ds_name):
    """HF 데이터셋 로드 후 train/test 인덱스 분할 (회귀 — stratify 없음)."""
    meta = DATASETS[ds_name]
    ds = load_dataset(meta['hf'], split='train')
    from sklearn.model_selection import train_test_split
    tr_idx, te_idx = train_test_split(np.arange(len(ds)), test_size=ML_TEST_SIZE,
                                      random_state=ML_SPLIT_SEED)
    return ds, meta, tr_idx, te_idx

def to_arrays(ds, meta, indices, sample_cap=None, random_state=1):
    """주어진 인덱스(예: train) 중 sample_cap개를 (images[np.uint8], targets[float])로."""
    indices = np.asarray(indices)
    if sample_cap and len(indices) > sample_cap:
        rng = np.random.RandomState(random_state)
        indices = indices[rng.permutation(len(indices))[:sample_cap]]
    images, targets = [], []
    for i in indices:
        ex = ds[int(i)]
        img = ex[meta['image_col']]
        if hasattr(img, 'convert'):
            img = img.convert('RGB')
        images.append(np.array(img, dtype=np.uint8))
        targets.append(float(ex[meta['target']]))
    return images, targets, indices

In [5]:
# ============================================================
# 회귀 모델 정의 (헤드 Linear(·,1)) + 학습/평가 (MSE → R²)
# ============================================================
import torch.nn as nn
import torchvision.models as tvm
import torchvision.transforms as T
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import r2_score

def get_model(model_name, in_channels=3):
    if model_name == 'ResNet18':
        m = tvm.resnet18(weights=tvm.ResNet18_Weights.IMAGENET1K_V1)
        m.fc = nn.Linear(m.fc.in_features, 1); return m
    if model_name == 'EfficientNetB0':
        m = tvm.efficientnet_b0(weights=tvm.EfficientNet_B0_Weights.IMAGENET1K_V1)
        m.classifier[1] = nn.Linear(m.classifier[1].in_features, 1); return m
    if model_name == 'MobileNetV3small':
        m = tvm.mobilenet_v3_small(weights=tvm.MobileNet_V3_Small_Weights.IMAGENET1K_V1)
        m.classifier[3] = nn.Linear(m.classifier[3].in_features, 1); return m
    if model_name == 'ViTTiny':
        import timm
        return timm.create_model('vit_tiny_patch16_224', pretrained=True, num_classes=1, in_chans=in_channels)
    if model_name == 'CNNSimple':
        return nn.Sequential(
            nn.Conv2d(in_channels, 32, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(32, 64, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(64, 128, 3, padding=1), nn.ReLU(), nn.AdaptiveAvgPool2d(1),
            nn.Flatten(), nn.Linear(128, 64), nn.ReLU(), nn.Linear(64, 1))
    raise ValueError(model_name)

class NumpyRegDataset(Dataset):
    def __init__(self, images, targets, transform):
        self.images, self.targets, self.transform = images, targets, transform
    def __len__(self): return len(self.images)
    def __getitem__(self, i):
        from PIL import Image
        img = self.images[i]
        if isinstance(img, np.ndarray):
            arr = img.astype(np.uint8)
            arr = arr.squeeze() if arr.ndim == 3 and arr.shape[-1] == 1 else arr
            img = Image.fromarray(arr) if arr.ndim == 2 else Image.fromarray(arr[..., :3])
        if img.mode != 'RGB': img = img.convert('RGB')
        return self.transform(img), float(self.targets[i])

def get_transform(size=IMAGE_SIZE):
    return T.Compose([T.Resize((size, size)), T.ToTensor(),
                      T.Normalize(mean=[0.485,0.456,0.406], std=[0.229,0.224,0.225])])

def train_eval(model, train_ds, test_ds, epochs=EPOCHS):
    model = model.to(device)
    opt = torch.optim.Adam(model.parameters(), lr=LR)
    crit = nn.MSELoss()
    tl = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=0)
    vl = DataLoader(test_ds, batch_size=BATCH_SIZE, num_workers=0)
    for _ in range(epochs):
        model.train()
        for x, y in tl:
            x, y = x.to(device), y.to(device).float().unsqueeze(1)
            opt.zero_grad(); loss = crit(model(x), y); loss.backward(); opt.step()
    model.eval(); preds, trues = [], []
    with torch.no_grad():
        for x, y in vl:
            preds.append(model(x.to(device)).cpu().numpy().ravel()); trues.append(np.asarray(y))
    r2 = r2_score(np.concatenate(trues), np.concatenate(preds))
    return r2, max(0.0, r2)
print('회귀 모델/학습 함수 정의 완료')

회귀 모델/학습 함수 정의 완료


In [6]:
# ============================================================
# 실험 목록 (clean baseline + POLLUTED_DIR 스캔) + 폴루션/clean 로더
# ============================================================
def load_polluted(ds_name, polluter, level):
    npz = np.load(f'{POLLUTED_DIR}/{ds_name}/{polluter}_{int(level*100)}/data.npz', allow_pickle=True)
    return list(npz['images']), npz['targets'].tolist()

# clean train/test (split 재현 — ML_SPLIT_SEED 고정이라 02와 동일 분할)
def load_clean(ds_name):
    ds, meta, tr_idx, te_idx = load_hf_split(ds_name)
    tr_i, tr_t, _ = to_arrays(ds, meta, tr_idx, sample_cap=SAMPLE_CAP, random_state=1)
    te_i, te_t, _ = to_arrays(ds, meta, te_idx, sample_cap=2000, random_state=1)  # test도 상한(RAM)
    return (tr_i, tr_t), (te_i, te_t)

experiments = []
for ds_name in DATASETS:
    experiments.append({'dataset': ds_name, 'polluter': 'none', 'level': 0.0})
    pol_dir = f'{POLLUTED_DIR}/{ds_name}'
    if os.path.isdir(pol_dir):
        for folder in sorted(os.listdir(pol_dir)):
            if not os.path.isfile(f'{pol_dir}/{folder}/data.npz'): continue
            parts = folder.rsplit('_', 1); level = int(parts[1]) / 100
            if level in POLLUTION_LEVELS:
                experiments.append({'dataset': ds_name, 'polluter': parts[0], 'level': level})
print(f'실험 {len(experiments)}건 × 모델 {len(MODEL_NAMES)} = {len(experiments)*len(MODEL_NAMES)}회 학습')

실험 52건 × 모델 2 = 104회 학습


In [ ]:
# ============================================================
# 학습 루프 (체크포인트 + clean test 고정) — RAM 절약: 반복마다 모델/데이터 해제
# ============================================================
import gc
from time import time
perf_path = f'{RESULTS_DIR}/model_performance_image_regression.csv'
if os.path.isfile(perf_path):
    df_perf = pd.read_csv(perf_path)
    existing = set(df_perf.apply(lambda r: f"{r['dataset']}|{r['polluter']}|{r['level']}|{r['model']}", axis=1))
    perf_rows = df_perf.to_dict('records'); print(f'기존 {len(perf_rows)}건 로드')
else:
    existing, perf_rows = set(), []

transform = get_transform()
clean_cache = {}
t0all = time(); completed = skipped = 0; errors = []

for i, exp in enumerate(experiments):
    ds_name = exp['dataset']
    if ds_name not in clean_cache:
        clean_cache[ds_name] = load_clean(ds_name)
    (tr_i, tr_t), (te_i, te_t) = clean_cache[ds_name]
    if exp['polluter'] == 'none':
        train_images, train_targets = tr_i, tr_t
    else:
        try:
            train_images, train_targets = load_polluted(ds_name, exp['polluter'], exp['level'])
        except Exception as e:
            print(f'load fail {ds_name}/{exp["polluter"]}: {e}'); continue
    train_ds = NumpyRegDataset(train_images, train_targets, transform)
    test_ds = NumpyRegDataset(te_i, te_t, transform)
    for model_name in MODEL_NAMES:
        key = f"{ds_name}|{exp['polluter']}|{exp['level']}|{model_name}"
        if key in existing:
            skipped += 1; continue
        model = None
        try:
            t0 = time(); model = get_model(model_name)
            r2, r2c = train_eval(model, train_ds, test_ds, epochs=EPOCHS)
            perf_rows.append({'dataset': ds_name, 'polluter': exp['polluter'], 'level': exp['level'],
                              'model': model_name, 'r2': round(r2, 4), 'r2_clipped': round(r2c, 4), 'epochs': EPOCHS})
            existing.add(key); completed += 1
            print(f'  [{i+1}/{len(experiments)}] {ds_name}/{exp["polluter"]}_{int(exp["level"]*100)}/{model_name} R2={r2:+.4f} ({time()-t0:.0f}s)')
        except Exception as e:
            errors.append((key, str(e))); print(f'  [{i+1}] {ds_name}/{model_name} ERROR: {e}')
        finally:
            del model; gc.collect(); torch.cuda.empty_cache()   # 모델/VRAM 해제 (반복 누적 방지)
        if (completed + skipped) % 5 == 0:
            pd.DataFrame(perf_rows).to_csv(perf_path, index=False)
    del train_ds, test_ds
    if exp['polluter'] != 'none':
        del train_images, train_targets
    gc.collect()

pd.DataFrame(perf_rows).to_csv(perf_path, index=False)
print(f'\n학습 완료: 완료={completed} 스킵={skipped} 에러={len(errors)} ({time()-t0all:.0f}s)')
print('--- 노트북 03 image regression 완료 → 04 실행 ---')

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md:   0%|          | 0.00/1.89k [00:00<?, ?B/s]

data/train-00000-of-00002.parquet:   0%|          | 0.00/362M [00:00<?, ?B/s]

data/train-00001-of-00002.parquet:   0%|          | 0.00/522M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/10135 [00:00<?, ? examples/s]

Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /root/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth


100%|██████████| 44.7M/44.7M [00:00<00:00, 150MB/s]


  [1/52] UTKFace/none_0/ResNet18 R2=+0.7633 (128s)
  [1/52] UTKFace/none_0/CNNSimple R2=+0.0197 (119s)
  [2/52] UTKFace/blur_10/ResNet18 R2=+0.8112 (129s)
  [2/52] UTKFace/blur_10/CNNSimple R2=+0.0132 (119s)
